# 06 – Pulizia dati: indicatori economici
Fonte: World Bank · PIL pro capite, disoccupazione giovanile, rimesse, popolazione

In [2]:
import pandas as pd
import glob

# I 4 indicatori: codice World Bank -> nome in italiano
indicatori = {
    "NY.GDP.PCAP.PP.KD": "PIL pro capite (PPP, $ costanti 2021)",
    "SL.UEM.1524.ZS": "Disoccupazione giovanile 15-24 (%)",
    "BX.TRF.PWKR.CD.DT": "Rimesse ricevute (US$ correnti)",
    "SP.POP.TOTL": "Popolazione totale"
}

# I 4 paesi del progetto (nomi in inglese, come nei dati Eurostat)
paesi = {"ALB": "Albania", "ITA": "Italy", "GRC": "Greece", "DEU": "Germany"}

In [3]:
tabelle = []

for codice, nome in indicatori.items():
    file = glob.glob(f"dataraw/API_{codice}_*.csv")[0]

    # skiprows=4: salto le prime 4 righe di intestazione
    d = pd.read_csv(file, skiprows=4)

    # Tengo solo i 4 paesi
    d = d[d["Country Code"].isin(paesi)]

    # Da formato "largo" (un anno per colonna) a formato "lungo" (una riga per anno)
    anni = [c for c in d.columns if c.isdigit()]
    d = d.melt(id_vars=["Country Code"], value_vars=anni,
               var_name="anno", value_name="valore")

    d["paese"] = d["Country Code"].map(paesi)
    d["indicatore"] = nome
    tabelle.append(d)
    print(nome, "->", len(d), "righe")

PIL pro capite (PPP, $ costanti 2021) -> 264 righe
Disoccupazione giovanile 15-24 (%) -> 264 righe
Rimesse ricevute (US$ correnti) -> 264 righe
Popolazione totale -> 264 righe


In [4]:
df = pd.concat(tabelle)

# Tolgo gli anni senza dato e converto l'anno in numero
df = df.dropna(subset=["valore"])
df["anno"] = df["anno"].astype(int)

df = df[["paese", "anno", "indicatore", "valore"]]
print("Righe:", len(df))
df.groupby("indicatore")["anno"].agg(["min", "max", "count"])

Righe: 742


,min,max,count
indicatore,,,
Disoccupazione giovanile 15-24 (%),1991,2025,140
"PIL pro capite (PPP, $ costanti 2021)",1990,2025,144
Popolazione totale,1960,2025,264
Rimesse ricevute (US$ correnti),1970,2025,194


In [5]:
df[df["anno"] == 2024].pivot_table(index="paese", columns="indicatore", values="valore").round(1)

indicatore,Disoccupazione giovanile 15-24 (%),"PIL pro capite (PPP, $ costanti 2021)",Popolazione totale,Rimesse ricevute (US$ correnti)
paese,,,,
Albania,25.6,21641.0,2377128.0,2.274441e+09
Germany,6.6,62654.6,83516593.0,2.216807e+10
Greece,22.4,37474.3,10405134.0,5.614475e+08
Italy,20.2,53284.9,58952704.0,1.214205e+10


In [6]:
df.to_csv("dataclean/indicatori_economici.csv", index=False)
print("File salvato!", len(df), "righe")

File salvato! 742 righe
